In [3]:
-# ============================================================
# FEDERAL URDU UNIVERSITY - FACULTY SUPPORT AGENT
# Google Colab Version
# ============================================================
#
# Features:
#   - No file upload
#   - Timetable stored directly in this notebook
#   - Search by day
#   - Search by section
#   - Search by instructor
#   - Search by room
#   - Search by subject
#   - Search today's schedule
#   - General questions through Gemini
#
# ============================================================


# ============================================================
# STEP 1: INSTALL REQUIRED PACKAGE
# ============================================================

!pip install -U google-genai


# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

from google import genai
from google.colab import userdata

import json
import re
from datetime import datetime


# ============================================================
# STEP 3: EMBED YOUR TIMETABLE JSON
# ============================================================
#
# IMPORTANT:
# Paste your COMPLETE JSON between the triple quotes.
#
# Do NOT use:
#     files.upload()
#
# Do NOT create data.json.
#
# ============================================================

TIMETABLE_JSON = r'''
{
  "universityTimeTable": [
    {
      "day": "Monday",
      "classes": [
        {
          "BS2A": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr. Muhammaad Sarim", "room": "Lab1"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr. Muhammaad Sarim", "room": "Lab1"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room1"},
            {"time": "11:30-12:20", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Aliza Ali", "room": "Room1"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room11"}
          ]
        },
        {
          "BS2B": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "10:40-11:30", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Aliza Ali", "room": "Room2"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room2"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room2"}
          ]
        },
        {
          "BS2C": [
            {"time": "09:00-09:50", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room13"},
            {"time": "09:50-10:40", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Aliza Ali", "room": "Room13"},
            {"time": "10:40-11:30", "subject": "Object Oriented Programming", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"},
            {"time": "11:30-12:20", "subject": "Object Oriented Programming", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room3"}
          ]
        },
        {
          "BS2D": [
            {"time": "10:40-11:30", "subject": "Quantitative Reasoning-II", "instructor": "Mr.Umair Waqas", "room": "Room3"},
            {"time": "11:30-12:20", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room3"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room11"}
          ]
        },
        {
          "BS2E": [
            {"time": "09:50-10:40", "subject": "Quantitative Reasoning-II", "instructor": "Mr.Umair Waqas", "room": "Room12"},
            {"time": "10:40-11:30", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room12"},
            {"time": "11:30-12:20", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room12"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room12"}
          ]
        },
        {
          "BS2F": [
            {"time": "09:50-10:40", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room11"},
            {"time": "10:40-11:30", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room11"},
            {"time": "11:30-12:20", "subject": "Quantitative Reasoning-II", "instructor": "Miss Meiraj Fatima", "room": "Room11"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room13"}
          ]
        },
        {
          "BS4A": [
            {"time": "09:50-10:40", "subject": "Linear Algebra", "instructor": "Dr. Kashif Bin Zaheer", "room": "Room1"},
            {"time": "10:40-11:30", "subject": "Database Management Systems", "instructor": "Dr. Khalid Sheikh", "room": "Lab1"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Miss. Naheed Azeem", "room": "Lab1"}
          ]
        },
        {
          "BS4B": [
            {"time": "10:40-11:30", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "12:20-13:10", "subject": "Linear Algebra", "instructor": "Mr. Farzand Ali", "room": "Room2"}
          ]
        },
        {
          "BS4C": [
            {"time": "10:40-11:30", "subject": "Linear Algebra", "instructor": "Miss Urusa Mehmood", "room": "Room12"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Room13"},
            {"time": "12:20-13:10", "subject": "Database Management Systems", "instructor": "Dr. Farhan Shafiq", "room": "Room13"}
          ]
        },
        {
          "BS4D": [
            {"time": "10:40-11:30", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Hall"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Miss Quratul Ain", "room": "Hall"},
            {"time": "12:20-13:10", "subject": "Linear Algebra", "instructor": "Dr. Amber Nihan Kashif", "room": "Hall"}
          ]
        },
        {
          "BS6A": [
            {"time": "12:20-13:10", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab1"},
            {"time": "14:00-14:50", "subject": "Natural Language Processing", "instructor": "Dr. Khalid Sheikh", "room": "Lab1"},
            {"time": "14:50-15:40", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab3"}
          ]
        },
        {
          "BS6B": [
            {"time": "12:20-13:10", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab2"},
            {"time": "14:00-14:50", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab2"},
            {"time": "14:50-15:40", "subject": "Natural Language Processing", "instructor": "Dr. Muhammad Sarim", "room": "Lab4"}
          ]
        },
        {
          "BS6C": [
            {"time": "11:30-12:20", "subject": "Compiler Construction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Lab3"},
            {"time": "12:20-13:10", "subject": "Natural Language Processing", "instructor": "Miss Quratul Ain", "room": "Lab3"},
            {"time": "14:00-14:50", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab3"},
            {"time": "14:50-15:40", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"}
          ]
        },
        {
          "BS6D": [
            {"time": "11:30-12:20", "subject": "Natural Language Processing", "instructor": "Mr. Muhammad Ahmed Ansari", "room": "Lab4"},
            {"time": "12:20-13:10", "subject": "Artificial Intelligence", "instructor": "Miss. Noshaba", "room": "Lab4"},
            {"time": "14:00-14:50", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Lab4"},
            {"time": "14:50-15:40", "subject": "Compiler Construction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Lab4"}
          ]
        },
        {
          "BS7": [
            {"time": "09:00-09:50", "subject": "Parallel and Distributed Computing", "instructor": "Dr. Muhammad Sarim", "room": "Lab4"},
            {"time": "09:50-10:40", "subject": "Data Science", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab4"},
            {"time": "10:40-11:30", "subject": "Grid and Cloud Computing", "instructor": "Dr. Uzma Afzal", "room": "Lab4"},
            {"time": "11:30-12:20", "subject": "Entrepreneurship", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Library"}
          ]
        },
        {
          "BS8A": [
            {"time": "14:00-14:50", "subject": "Human Computer Interaction", "instructor": "Dr. Khalid Sheikh", "room": "Hall"},
            {"time": "14:50-15:40", "subject": "Wireless Networks", "instructor": "Miss. Naheed Azeem", "room": "Hall"}
          ]
        },
        {
          "BS8B": [
            {"time": "14:00-14:50", "subject": "Human Computer Interaction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Room12"},
            {"time": "14:50-15:40", "subject": "Wireless Networks", "instructor": "Mrs. Asma Nisar", "room": "Room12"}
          ]
        },
        {
          "BS8C": [
            {"time": "14:00-14:50", "subject": "Wireless Networks", "instructor": "Mrs. Asma Nisar", "room": "Room13"},
            {"time": "14:50-15:40", "subject": "Human Computer Interaction", "instructor": "Mrs.Salwa Iqbal", "room": "Room13"}
          ]
        }
      ]
    },
    {
      "day": "Tuesday",
      "classes": [
        {
          "BS2A": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr. Muhammaad Sarim", "room": "Lab1"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr. Muhammaad Sarim", "room": "Lab1"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room1"},
            {"time": "11:30-12:20", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Aliza Ali", "room": "Room1"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2B": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "10:40-11:30", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Aliza Ali", "room": "Room2"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room2"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2C": [
            {"time": "09:00-09:50", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room13"},
            {"time": "09:50-10:40", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Aliza Ali", "room": "Room13"},
            {"time": "10:40-11:30", "subject": "Object Oriented Programming", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"},
            {"time": "11:30-12:20", "subject": "Object Oriented Programming", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2D": [
            {"time": "09:00-09:50", "subject": "Quantitative Reasoning-II", "instructor": "Mr.Umair Waqas", "room": "Room2"},
            {"time": "09:50-10:40", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room2"},
            {"time": "10:40-11:30", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room2"},
            {"time": "11:30-12:20", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room2"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2E": [
            {"time": "10:40-11:30", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room3"},
            {"time": "11:30-12:20", "subject": "Quantitative Reasoning-II", "instructor": "Mr.Umair Waqas", "room": "Room3"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2F": [
            {"time": "09:50-10:40", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room11"},
            {"time": "10:40-11:30", "subject": "Quantitative Reasoning-II", "instructor": "Miss Meiraj Fatima", "room": "Room11"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room11"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS4A": [
            {"time": "09:50-10:40", "subject": "Linear Algebra", "instructor": "Dr. Kashif Bin Zaheer", "room": "Room12"},
            {"time": "10:40-11:30", "subject": "Theory of Automata & Formal languages", "instructor": "Miss. Naheed Azeem", "room": "Lab1"},
            {"time": "11:30-12:20", "subject": "Database Management Systems", "instructor": "Dr. Khalid Sheikh", "room": "Lab1"}
          ]
        },
        {
          "BS4B": [
            {"time": "09:50-10:40", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Lab2"},
            {"time": "10:40-11:30", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Linear Algebra", "instructor": "Mr. Farzand Ali", "room": "Room1"}
          ]
        },
        {
          "BS4C": [
            {"time": "10:40-11:30", "subject": "Linear Algebra", "instructor": "Miss Urusa Mehmood", "room": "Room12"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Hall"},
            {"time": "12:20-13:10", "subject": "Database Management Systems", "instructor": "Dr. Farhan Shafiq", "room": "Hall"}
          ]
        },
        {
          "BS4D": [
            {"time": "09:50-10:40", "subject": "Linear Algebra", "instructor": "Dr. Amber Nihan Kashif", "room": "Hall"},
            {"time": "10:40-11:30", "subject": "Theory of Automata & Formal languages", "instructor": "Miss Quratul Ain", "room": "Hall"},
            {"time": "11:30-12:20", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Lab2"}
          ]
        },
        {
          "BS6A": [
            {"time": "12:20-13:10", "subject": "Compiler Construction", "instructor": "Miss. Naheed Azeem", "room": "Lab1"},
            {"time": "14:00-14:50", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab1"},
            {"time": "14:50-15:40", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab1"},
            {"time": "15:40-16:30", "subject": "Natural Language Processing", "instructor": "Dr. Khalid Sheikh", "room": "Lab1"}
          ]
        },
        {
          "BS6B": [
            {"time": "12:20-13:10", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab2"},
            {"time": "14:00-14:50", "subject": "Compiler Construction", "instructor": "Miss. Naheed Azeem", "room": "Lab2"},
            {"time": "14:50-15:40", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab2"}
          ]
        },
        {
          "BS6C": [
            {"time": "12:20-13:10", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab3"},
            {"time": "14:00-14:50", "subject": "Compiler Construction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Lab3"},
            {"time": "14:50-15:40", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"}
          ]
        },
        {
          "BS6D": [
            {"time": "12:20-13:10", "subject": "Artificial Intelligence", "instructor": "Miss. Noshaba", "room": "Lab4"},
            {"time": "14:00-14:50", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Lab4"},
            {"time": "14:50-15:40", "subject": "Compiler Construction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Lab4"}
          ]
        },
        {
          "BS7": [
            {"time": "14:00-14:50", "subject": "Human Computer Interaction", "instructor": "Dr. Khalid Sheikh", "room": "Room13"},
            {"time": "14:50-15:40", "subject": "Wireless Networks", "instructor": "Miss. Naheed Azeem", "room": "Room13"}
          ]
        },
        {
          "BS8A": [
            {"time": "12:20-13:10", "subject": "Human Computer Interaction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Room12"},
            {"time": "14:00-14:50", "subject": "Wireless Networks", "instructor": "Mrs. Asma Nisar", "room": "Room11"}
          ]
        },
        {
          "BS8B": [
            {"time": "12:20-13:10", "subject": "Wireless Networks", "instructor": "Mrs. Asma Nisar", "room": "Room11"},
            {"time": "14:00-14:50", "subject": "Human Computer Interaction", "instructor": "Mrs.Salwa Iqbal", "room": "Room12"}
          ]
        }
      ]
    },
    {
      "day": "Wednesday",
      "classes": [
        {
          "BS2A": [
            {"time": "09:00-09:50", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room1"},
            {"time": "09:50-10:40", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room1"},
            {"time": "10:40-11:30", "subject": "Object Oriented Programming", "instructor": "Dr. Muhammaad Sarim", "room": "Room1"},
            {"time": "11:30-12:20", "subject": "Quantitative Reasoning-II", "instructor": "Mrs. Asma Nisar", "room": "Lab1"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2B": [
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr Shazia Usmani", "room": "Lab2"},
            {"time": "10:40-11:30", "subject": "Quantitative Reasoning-II", "instructor": "Mrs. Asma Nisar", "room": "Room2"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room2"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2C": [
            {"time": "09:50-10:40", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room3"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Lab4"},
            {"time": "11:30-12:20", "subject": "Object Oriented Programming", "instructor": "Dr. Farhan Shafiq", "room": "Room3"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2D": [
            {"time": "10:40-11:30", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Binish Siddiqui", "room": "Room11"},
            {"time": "11:30-12:20", "subject": "Object Oriented Programming", "instructor": "Dr. Khalid Shaikh", "room": "Room11"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2E": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr. Shazia Usmani", "room": "Lab4"},
            {"time": "09:50-10:40", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Binish Siddiqui", "room": "Room2"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room12"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room12"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS2F": [
            {"time": "10:40-11:30", "subject": "Object Oriented Programming", "instructor": "Mrs. Salwa Iqbal", "room": "Lab1"},
            {"time": "11:30-12:20", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Binish Siddiqui", "room": "Room13"},
            {"time": "12:20-13:10", "subject": "Pre Calculus", "instructor": "Miss. Humaira Zulfiqar", "room": "Room1"}
          ]
        },
        {
          "BS4A": [
            {"time": "09:00-09:50", "subject": "Design and Analysis of Algorithm", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Room12"},
            {"time": "09:50-10:40", "subject": "Linear Algebra", "instructor": "Dr. Kashif Bin Zaheer", "room": "Room12"},
            {"time": "10:40-11:30", "subject": "Database Management Systems", "instructor": "Dr. Khalid Sheikh", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Miss. Naheed Azeem", "room": "Lab2"},
            {"time": "12:20-13:10", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Room12"}
          ]
        },
        {
          "BS4B": [
            {"time": "09:00-09:50", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Room13"},
            {"time": "09:50-10:40", "subject": "Design and Analysis of Algorithm", "instructor": "Dr. Syed Akhtar Raza", "room": "Room13"},
            {"time": "10:40-11:30", "subject": "Linear Algebra", "instructor": "Mr. Farzand Ali", "room": "Room13"},
            {"time": "11:30-12:20", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Bbb", "room": "Room13"},
            {"time": "12:20-13:10", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Lab2"}
          ]
        },
        {
          "BS4C": [
            {"time": "09:00-09:50", "subject": "Design and Analysis of Algorithm", "instructor": "Miss. Noshaba", "room": "Hall"},
            {"time": "09:50-10:40", "subject": "Linear Algebra", "instructor": "Miss Urusa Mehmood", "room": "Hall"},
            {"time": "10:40-11:30", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Ccc", "room": "Hall"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Lab1"},
            {"time": "12:20-13:10", "subject": "Database Management Systems", "instructor": "Dr. Farhan Shafiq", "room": "Lab1"}
          ]
        },
        {
          "BS4D": [
            {"time": "09:00-09:50", "subject": "Design and Analysis of Algorithm", "instructor": "Mr. Muhammad Ahmed Ansari", "room": "Room11"},
            {"time": "09:50-10:40", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Ddd", "room": "Room11"},
            {"time": "10:40-11:30", "subject": "Linear Algebra", "instructor": "Dr. Amber Nihan Kashif", "room": "Room11"},
            {"time": "11:30-12:20", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Lab3"},
            {"time": "12:20-13:10", "subject": "Theory of Automata & Formal languages", "instructor": "Miss Quratul Ain", "room": "Lab3"}
          ]
        },
        {
          "BS6A": [
            {"time": "12:20-13:10", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Room2"},
            {"time": "14:00-14:50", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Room2"},
            {"time": "14:50-15:40", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab1"}
          ]
        },
        {
          "BS6B": [
            {"time": "12:20-13:10", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab4"},
            {"time": "14:00-14:50", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Bbb", "room": "Room1"},
            {"time": "14:50-15:40", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Room1"}
          ]
        },
        {
          "BS6C": [
            {"time": "12:20-13:10", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Room3"},
            {"time": "14:00-14:50", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab3"},
            {"time": "14:50-15:40", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Ccc", "room": "Lab3"}
          ]
        },
        {
          "BS6D": [
            {"time": "12:20-13:10", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Ddd", "room": "Room11"},
            {"time": "14:00-14:50", "subject": "Artificial Intelligence", "instructor": "Miss. Noshaba", "room": "Lab4"},
            {"time": "14:50-15:40", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Lab4"}
          ]
        },
        {
          "BS7": [
            {"time": "14:00-14:50", "subject": "Human Computer Interaction", "instructor": "Dr. Khalid Sheikh", "room": "Room13"},
            {"time": "14:50-15:40", "subject": "Wireless Networks", "instructor": "Miss. Naheed Azeem", "room": "Room13"}
          ]
        },
        {
          "BS8A": [
            {"time": "14:00-14:50", "subject": "Human Computer Interaction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Room11"},
            {"time": "14:50-15:40", "subject": "Wireless Networks", "instructor": "Mrs. Asma Nisar", "room": "Room11"}
          ]
        },
        {
          "BS8B": [
            {"time": "14:00-14:50", "subject": "Wireless Networks", "instructor": "Mrs. Asma Nisar", "room": "Room12"},
            {"time": "14:50-15:40", "subject": "Human Computer Interaction", "instructor": "Mrs.Salwa Iqbal", "room": "Room12"}
          ]
        }
      ]
    },
    {
      "day": "Thursday",
      "classes": [
        {
          "BS2A": [
            {"time": "09:00-09:50", "subject": "Quantitative Reasoning-II", "instructor": "Mrs. Asma Nisar", "room": "Room1"},
            {"time": "09:50-10:40", "subject": "Pakistan Studies", "instructor": "Mr. Waheed Murad 2", "room": "Room1"},
            {"time": "10:40-11:30", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room1"},
            {"time": "11:30-12:20", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room1"}
          ]
        },
        {
          "BS2B": [
            {"time": "09:00-09:50", "subject": "Pakistan Studies", "instructor": "Mr. Waheed Murad 2", "room": "Room2"},
            {"time": "09:50-10:40", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room2"},
            {"time": "10:40-11:30", "subject": "Quantitative Reasoning-II", "instructor": "Mrs. Asma Nisar", "room": "Room2"},
            {"time": "11:30-12:20", "subject": "Quantitative Reasoning-II", "instructor": "Mrs. Asma Nisar", "room": "Room2"}
          ]
        },
        {
          "BS2C": [
            {"time": "09:50-10:40", "subject": "Quantitative Reasoning-II", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Room3"},
            {"time": "10:40-11:30", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room3"},
            {"time": "11:30-12:20", "subject": "Pakistan Studies", "instructor": "Miss. Zamarood Banoo 2", "room": "Room3"}
          ]
        },
        {
          "BS2D": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr. Khalid Shaikh", "room": "Lab1"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr. Khalid Shaikh", "room": "Lab1"},
            {"time": "10:40-11:30", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Binish Siddiqui", "room": "Library"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Library"}
          ]
        },
        {
          "BS2E": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr. Shazia Usmani", "room": "Lab2"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr. Shazia Usmani", "room": "Lab2"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Binish Siddiqui", "room": "Room13"}
          ]
        },
        {
          "BS2F": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Mrs. Salwa Iqbal", "room": "Lab3"},
            {"time": "09:50-10:40", "subject": "Arts and Humanities (Urdu)", "instructor": "Dr Binish Siddiqui", "room": "Lab3"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room11"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room11"},
            {"time": "12:20-13:10", "subject": "Object Oriented Programming", "instructor": "Mrs. Salwa Iqbal", "room": "Room11"}
          ]
        },
        {
          "BS4A": [
            {"time": "09:00-09:50", "subject": "Design and Analysis of Algorithm", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Lab4"},
            {"time": "09:50-10:40", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Lab4"},
            {"time": "10:40-11:30", "subject": "Database Management Systems", "instructor": "Dr. Khalid Sheikh", "room": "Lab4"}
          ]
        },
        {
          "BS4B": [
            {"time": "09:00-09:50", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Hall"},
            {"time": "09:50-10:40", "subject": "Design and Analysis of Algorithm", "instructor": "Dr. Syed Akhtar Raza", "room": "Hall"},
            {"time": "10:40-11:30", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Bbb", "room": "Hall"}
          ]
        },
        {
          "BS4C": [
            {"time": "09:00-09:50", "subject": "Design and Analysis of Algorithm", "instructor": "Miss. Noshaba", "room": "Room12"},
            {"time": "09:50-10:40", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Ccc", "room": "Room12"},
            {"time": "10:40-11:30", "subject": "Database Management Systems", "instructor": "Dr. Farhan Shafiq", "room": "Room12"}
          ]
        },
        {
          "BS4D": [
            {"time": "09:00-09:50", "subject": "Database Management Systems", "instructor": "Dr Shazia Usmani", "room": "Room13"},
            {"time": "09:50-10:40", "subject": "Design and Analysis of Algorithm", "instructor": "Mr. Muhammad Ahmed Ansari", "room": "Room13"},
            {"time": "10:40-11:30", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Ddd", "room": "Room13"}
          ]
        },
        {
          "BS6A": [
            {"time": "10:40-11:30", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab1"},
            {"time": "11:30-12:20", "subject": "Compiler Construction", "instructor": "Miss. Naheed Azeem", "room": "Lab1"},
            {"time": "12:20-13:10", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Lab1"},
            {"time": "14:00-14:50", "subject": "Natural Language Processing", "instructor": "Dr. Khalid Sheikh", "room": "Lab1"},
            {"time": "14:50-15:40", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab3"}
          ]
        },
        {
          "BS6B": [
            {"time": "10:40-11:30", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Bbb", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Compiler Construction", "instructor": "Miss. Naheed Azeem", "room": "Lab2"},
            {"time": "14:00-14:50", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab2"},
            {"time": "14:50-15:40", "subject": "Natural Language Processing", "instructor": "Dr. Muhammad Sarim", "room": "Lab4"}
          ]
        },
        {
          "BS6C": [
            {"time": "10:40-11:30", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Lab3"},
            {"time": "11:30-12:20", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Ccc", "room": "Lab3"},
            {"time": "14:00-14:50", "subject": "Natural Language Processing", "instructor": "Miss Quratul Ain", "room": "Lab3"},
            {"time": "14:50-15:40", "subject": "Artificial Intelligence", "instructor": "Dr Shazia Usmani", "room": "Lab3"}
          ]
        },
        {
          "BS6D": [
            {"time": "10:40-11:30", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Ddd", "room": "Room1"},
            {"time": "11:30-12:20", "subject": "Numerical and Symbolic Computing", "instructor": "Dr. Farhan Shafiq", "room": "Room13"},
            {"time": "14:00-14:50", "subject": "Natural Language Processing", "instructor": "Mr. Muhammad Ahmed Ansari", "room": "Room13"},
            {"time": "14:50-15:40", "subject": "Artificial Intelligence", "instructor": "Miss. Noshaba", "room": "Room13"}
          ]
        },
        {
          "BS7": [
            {"time": "10:40-11:30", "subject": "Data Science", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab4"},
            {"time": "11:30-12:20", "subject": "Parallel and Distributed Computing", "instructor": "Dr. Muhammad Sarim", "room": "Lab4"},
            {"time": "14:00-14:50", "subject": "Grid and Cloud Computing", "instructor": "Dr. Uzma Afzal", "room": "Lab4"},
            {"time": "14:50-15:40", "subject": "Entrepreneurship", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Lab4"}
          ]
        }
      ]
    },
    {
      "day": "Friday",
      "classes": [
        {
          "BS2A": [
            {"time": "09:00-09:50", "subject": "Quantitative Reasoning-II", "instructor": "Mrs. Asma Nisar", "room": "Room1"},
            {"time": "09:50-10:40", "subject": "Pakistan Studies", "instructor": "Mr. Waheed Murad 2", "room": "Room1"}
          ]
        },
        {
          "BS2B": [
            {"time": "09:00-09:50", "subject": "Pakistan Studies", "instructor": "Mr. Waheed Murad 2", "room": "Room2"},
            {"time": "09:50-10:40", "subject": "Fehm-e-Quran-1 (Form Muslim)", "instructor": "Dr. Mohsin Ali", "room": "Room2"}
          ]
        },
        {
          "BS2C": [
            {"time": "09:50-10:40", "subject": "Quantitative Reasoning-II", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Room11"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room11"}
          ]
        },
        {
          "BS2D": [
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr. Khalid Shaikh", "room": "Lab1"},
            {"time": "10:40-11:30", "subject": "Object Oriented Programming", "instructor": "Dr. Khalid Shaikh", "room": "Lab1"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room3"},
            {"time": "12:20-13:10", "subject": "Digital Logic Design", "instructor": "Mr. Majid Iqbal", "room": "Room3"}
          ]
        },
        {
          "BS2E": [
            {"time": "09:00-09:50", "subject": "Object Oriented Programming", "instructor": "Dr. Shazia Usmani", "room": "Lab2"},
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Dr. Shazia Usmani", "room": "Lab2"},
            {"time": "10:40-11:30", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room2"}
          ]
        },
        {
          "BS2F": [
            {"time": "09:50-10:40", "subject": "Object Oriented Programming", "instructor": "Mrs. Salwa Iqbal", "room": "Lab3"},
            {"time": "10:40-11:30", "subject": "Object Oriented Programming", "instructor": "Mrs. Salwa Iqbal", "room": "Lab3"},
            {"time": "11:30-12:20", "subject": "Digital Logic Design", "instructor": "Miss Quratul Ain", "room": "Room2"}
          ]
        },
        {
          "BS4A": [
            {"time": "09:00-09:50", "subject": "Design and Analysis of Algorithm", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Lab4"},
            {"time": "09:50-10:40", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Library"},
            {"time": "10:40-11:30", "subject": "Theory of Automata & Formal languages", "instructor": "Miss. Naheed Azeem", "room": "Room12"}
          ]
        },
        {
          "BS4B": [
            {"time": "09:50-10:40", "subject": "Design and Analysis of Algorithm", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab4"},
            {"time": "10:40-11:30", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Bbb", "room": "Lab4"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Lab4"}
          ]
        },
        {
          "BS4C": [
            {"time": "09:00-09:50", "subject": "Theory of Automata & Formal languages", "instructor": "Mrs.Salwa Iqbal", "room": "Room12"},
            {"time": "09:50-10:40", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Ccc", "room": "Room12"},
            {"time": "10:40-11:30", "subject": "Design and Analysis of Algorithm", "instructor": "Miss. Noshaba", "room": "Room1"}
          ]
        },
        {
          "BS4D": [
            {"time": "10:40-11:30", "subject": "Accounting", "instructor": "Mr. Mrs. Miss. Ddd", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Theory of Automata & Formal languages", "instructor": "Miss Quratul Ain", "room": "Room13"},
            {"time": "12:20-13:10", "subject": "Design and Analysis of Algorithm", "instructor": "Mr. Muhammad Ahmed Ansari", "room": "Room13"}
          ]
        },
        {
          "BS6A": [
            {"time": "10:40-11:30", "subject": "Compiler Construction", "instructor": "Miss. Naheed Azeem", "room": "Lab2"},
            {"time": "11:30-12:20", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Room1"}
          ]
        },
        {
          "BS6B": [
            {"time": "09:00-09:50", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Bbb", "room": "Hall"},
            {"time": "09:50-10:40", "subject": "Compiler Construction", "instructor": "Miss. Naheed Azeem", "room": "Hall"},
            {"time": "10:40-11:30", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Hall"},
            {"time": "11:30-12:20", "subject": "Natural Language Processing", "instructor": "Dr. Muhammad Sarim", "room": "Hall"}
          ]
        },
        {
          "BS6C": [
            {"time": "09:50-10:40", "subject": "Natural Language Processing", "instructor": "Miss Quratul Ain", "room": "Room3"},
            {"time": "10:40-11:30", "subject": "Compiler Construction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Room3"},
            {"time": "11:30-12:20", "subject": "Artificial Intelligence", "instructor": "Dr. Uzma Afzal", "room": "Lab1"},
            {"time": "12:20-13:10", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Ccc", "room": "Room3"}
          ]
        },
        {
          "BS6D": [
            {"time": "10:40-11:30", "subject": "Natural Language Processing", "instructor": "Mr. Muhammad Ahmed Ansari", "room": "Library"},
            {"time": "11:30-12:20", "subject": "Professional Practices", "instructor": "Mr. Mrs. Miss. Ddd", "room": "Room12"},
            {"time": "12:20-13:10", "subject": "Compiler Construction", "instructor": "Mr. Sheikh Kashif Rafat", "room": "Room12"}
          ]
        },
        {
          "BS7": [
            {"time": "09:50-10:40", "subject": "Grid and Cloud Computing", "instructor": "Dr. Uzma Afzal", "room": "Room13"},
            {"time": "10:40-11:30", "subject": "Parallel and Distributed Computing", "instructor": "Dr. Muhammad Sarim", "room": "Room13"},
            {"time": "11:30-12:20", "subject": "Entrepreneurship", "instructor": "Mr. Mrs. Miss. Aaa", "room": "Lab3"},
            {"time": "12:20-13:10", "subject": "Data Science", "instructor": "Dr. Syed Akhtar Raza", "room": "Lab3"}
          ]
        }
      ]
    }
  ]
}
'''


# ============================================================
# STEP 4: LOAD TIMETABLE
# ============================================================

try:

    timetable_data = json.loads(TIMETABLE_JSON)

    # Check that the expected structure exists
    if "universityTimeTable" not in timetable_data:
        raise ValueError(
            "JSON must contain 'universityTimeTable'"
        )

    print("=" * 70)
    print("🎓 FEDERAL URDU UNIVERSITY")
    print("Faculty Support Agent")
    print("=" * 70)

    print(
        f"✅ Timetable loaded successfully!"
    )

    print(
        f"📅 Number of days: "
        f"{len(timetable_data['universityTimeTable'])}"
    )

except json.JSONDecodeError as e:

    print("❌ Invalid JSON!")
    print()
    print("Please check your TIMETABLE_JSON.")
    print()
    print(f"JSON error: {e}")

    raise

except Exception as e:

    print("❌ Error loading timetable:")
    print(e)

    raise


# ============================================================
# STEP 5: DEBUG TIMETABLE STRUCTURE
# ============================================================

print()
print("=" * 70)
print("📊 TIMETABLE STRUCTURE")
print("=" * 70)

first_day = timetable_data["universityTimeTable"][0]

print(f"First day: {first_day.get('day')}")
print(f"Classes type: {type(first_day.get('classes'))}")
print(f"Number of sections: {len(first_day.get('classes', []))}")

if first_day.get("classes"):

    first_class = first_day["classes"][0]

    print(
        f"First section: "
        f"{list(first_class.keys())[0]}"
    )

print("=" * 70)


# ============================================================
# STEP 6: NORMALIZATION HELPER
# ============================================================

def normalize_text(text):
    """
    Normalize text to make searching easier.

    Example:
        'Dr. Khalid Sheikh'
        becomes
        'dr khalid sheikh'
    """

    if text is None:
        return ""

    text = str(text).lower()

    # Remove punctuation
    text = re.sub(r"[^\w\s-]", " ", text)

    # Replace multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# STEP 7: FIND BY DAY
# ============================================================

def find_by_day(day_name):
    """
    Find timetable for a specific day.
    """

    search_day = normalize_text(day_name)

    for day in timetable_data["universityTimeTable"]:

        current_day = normalize_text(
            day.get("day", "")
        )

        if current_day == search_day:

            return day

    return None


# ============================================================
# STEP 8: FIND BY SECTION
# ============================================================

def find_by_section(section_name):
    """
    Find all classes for a section.

    Example:
        BS2A
        BS4B
        BS6C
        BS7
        BS8A
    """

    results = []

    search_section = normalize_text(
        section_name
    )

    for day in timetable_data["universityTimeTable"]:

        day_classes = []

        for class_item in day.get("classes", []):

            for section, schedule in class_item.items():

                if normalize_text(section) == search_section:

                    day_classes.append({
                        "section": section,
                        "schedule": schedule
                    })

        if day_classes:

            results.append({
                "day": day["day"],
                "classes": day_classes
            })

    return results


# ============================================================
# STEP 9: FIND BY INSTRUCTOR
# ============================================================

def find_by_instructor(instructor_name):
    """
    Find all classes taught by an instructor.

    Partial matching is supported.

    Example:
        Khalid Sheikh
        Uzma Afzal
        Naheed Azeem
    """

    results = []

    search_name = normalize_text(
        instructor_name
    )

    for day in timetable_data["universityTimeTable"]:

        day_classes = []

        for class_item in day.get("classes", []):

            for section, schedule in class_item.items():

                for slot in schedule:

                    instructor = slot.get(
                        "instructor",
                        ""
                    )

                    if search_name in normalize_text(
                        instructor
                    ):

                        day_classes.append({
                            "section": section,
                            "time": slot.get(
                                "time",
                                "N/A"
                            ),
                            "subject": slot.get(
                                "subject",
                                "N/A"
                            ),
                            "instructor": instructor,
                            "room": slot.get(
                                "room",
                                "N/A"
                            )
                        })

        if day_classes:

            results.append({
                "day": day["day"],
                "classes": day_classes
            })

    return results


# ============================================================
# STEP 10: FIND BY ROOM
# ============================================================

def find_by_room(room_name):
    """
    Find all classes in a room.

    Example:
        Lab1
        Lab2
        Room1
        Room13
        Hall
        Library
    """

    results = []

    search_room = normalize_text(
        room_name
    )

    for day in timetable_data["universityTimeTable"]:

        day_classes = []

        for class_item in day.get("classes", []):

            for section, schedule in class_item.items():

                for slot in schedule:

                    room = slot.get(
                        "room",
                        ""
                    )

                    if search_room in normalize_text(
                        room
                    ):

                        day_classes.append({
                            "section": section,
                            "time": slot.get(
                                "time",
                                "N/A"
                            ),
                            "subject": slot.get(
                                "subject",
                                "N/A"
                            ),
                            "instructor": slot.get(
                                "instructor",
                                "N/A"
                            ),
                            "room": room
                        })

        if day_classes:

            results.append({
                "day": day["day"],
                "classes": day_classes
            })

    return results


# ============================================================
# STEP 11: FIND BY SUBJECT
# ============================================================

def find_by_subject(subject_name):
    """
    Find all classes for a subject.

    Example:
        Artificial Intelligence
        Database Management Systems
        Object Oriented Programming
        NLP
    """

    results = []

    search_subject = normalize_text(
        subject_name
    )

    # Subject aliases
    aliases = {
        "ai": "artificial intelligence",
        "nlp": "natural language processing",
        "oop": "object oriented programming",
        "db": "database management systems",
        "database": "database management systems",
        "dld": "digital logic design"
    }

    if search_subject in aliases:

        search_subject = aliases[
            search_subject
        ]

    for day in timetable_data["universityTimeTable"]:

        day_classes = []

        for class_item in day.get("classes", []):

            for section, schedule in class_item.items():

                for slot in schedule:

                    subject = slot.get(
                        "subject",
                        ""
                    )

                    if search_subject in normalize_text(
                        subject
                    ):

                        day_classes.append({
                            "section": section,
                            "time": slot.get(
                                "time",
                                "N/A"
                            ),
                            "subject": subject,
                            "instructor": slot.get(
                                "instructor",
                                "N/A"
                            ),
                            "room": slot.get(
                                "room",
                                "N/A"
                            )
                        })

        if day_classes:

            results.append({
                "day": day["day"],
                "classes": day_classes
            })

    return results


# ============================================================
# STEP 12: TODAY'S SCHEDULE
# ============================================================

def find_today_schedule():

    today = datetime.now().strftime(
        "%A"
    )

    return find_by_day(today)


# ============================================================
# STEP 13: FORMAT SCHEDULE
# ============================================================

def format_schedule(results):

    if not results:

        return "No classes found."

    output = []

    # --------------------------------------------------------
    # Single day
    # --------------------------------------------------------

    if isinstance(results, dict):

        if "day" in results:

            output.append(
                f"\n📅 {results['day']} Schedule:"
            )

            for class_item in results.get(
                "classes",
                []
            ):

                for section, schedule in class_item.items():

                    output.append(
                        f"\n📚 {section}"
                    )

                    for slot in schedule:

                        output.append(
                            f"  ⏰ {slot.get('time', 'N/A')}"
                        )

                        output.append(
                            f"     📖 {slot.get('subject', 'N/A')}"
                        )

                        output.append(
                            f"     👨‍🏫 {slot.get('instructor', 'N/A')}"
                        )

                        output.append(
                            f"     🏫 {slot.get('room', 'N/A')}"
                        )

    # --------------------------------------------------------
    # Multiple days
    # --------------------------------------------------------

    elif isinstance(results, list):

        for day_result in results:

            output.append(
                f"\n📅 {day_result['day']}"
            )

            for class_item in day_result.get(
                "classes",
                []
            ):

                # ------------------------------------------------
                # Flat result
                # ------------------------------------------------

                if (
                    isinstance(class_item, dict)
                    and "section" in class_item
                    and "time" in class_item
                ):

                    output.append(
                        f"\n  📚 {class_item['section']}"
                    )

                    output.append(
                        f"     ⏰ {class_item.get('time', 'N/A')}"
                    )

                    output.append(
                        f"     📖 {class_item.get('subject', 'N/A')}"
                    )

                    output.append(
                        f"     👨‍🏫 {class_item.get('instructor', 'N/A')}"
                    )

                    output.append(
                        f"     🏫 {class_item.get('room', 'N/A')}"
                    )

                # ------------------------------------------------
                # Section result
                # ------------------------------------------------

                elif (
                    isinstance(class_item, dict)
                    and "section" in class_item
                    and "schedule" in class_item
                ):

                    output.append(
                        f"\n  📚 {class_item['section']}"
                    )

                    for slot in class_item[
                        "schedule"
                    ]:

                        output.append(
                            f"     ⏰ {slot.get('time', 'N/A')}"
                        )

                        output.append(
                            f"     📖 {slot.get('subject', 'N/A')}"
                        )

                        output.append(
                            f"     👨‍🏫 {slot.get('instructor', 'N/A')}"
                        )

                        output.append(
                            f"     🏫 {slot.get('room', 'N/A')}"
                        )

    return "\n".join(output)


# ============================================================
# STEP 14: GET ALL SECTIONS
# ============================================================

def get_all_sections():

    sections = set()

    for day in timetable_data[
        "universityTimeTable"
    ]:

        for class_item in day.get(
            "classes",
            []
        ):

            for section in class_item.keys():

                sections.add(section)

    return sorted(sections)


# ============================================================
# STEP 15: GET ALL INSTRUCTORS
# ============================================================

def get_all_instructors():

    instructors = set()

    for day in timetable_data[
        "universityTimeTable"
    ]:

        for class_item in day.get(
            "classes",
            []
        ):

            for section, schedule in class_item.items():

                for slot in schedule:

                    instructor = slot.get(
                        "instructor"
                    )

                    if instructor:

                        instructors.add(
                            instructor
                        )

    return sorted(instructors)


# ============================================================
# STEP 16: GET ALL SUBJECTS
# ============================================================

def get_all_subjects():

    subjects = set()

    for day in timetable_data[
        "universityTimeTable"
    ]:

        for class_item in day.get(
            "classes",
            []
        ):

            for section, schedule in class_item.items():

                for slot in schedule:

                    subject = slot.get(
                        "subject"
                    )

                    if subject:

                        subjects.add(
                            subject
                        )

    return sorted(subjects)


# ============================================================
# STEP 17: INITIALIZE GEMINI
# ============================================================

try:

    api_key = userdata.get(
        "geminiSecretKey"
    )

    if not api_key:

        raise ValueError(
            "Gemini API key not found."
        )

    client = genai.Client(
        api_key=api_key
    )

    MODEL = "models/gemini-3.5-flash"

    print()
    print(
        "✅ Gemini AI initialized successfully!"
    )

except Exception as e:

    print()
    print(
        f"⚠️ Gemini initialization failed: {e}"
    )

    print(
        "⚠️ Agent will use rule-based mode."
    )

    client = None


# ============================================================
# STEP 18: EXTRACT SECTION FROM QUERY
# ============================================================

def detect_section(query):

    normalized = normalize_text(
        query
    ).replace(" ", "")

    sections = get_all_sections()

    for section in sections:

        if normalize_text(
            section
        ).replace(" ", "") in normalized:

            return section

    return None


# ============================================================
# STEP 19: DETECT DAY
# ============================================================

def detect_day(query):

    normalized = normalize_text(
        query
    )

    days = [
        "monday",
        "tuesday",
        "wednesday",
        "thursday",
        "friday",
        "today"
    ]

    for day in days:

        if day in normalized:

            return day

    return None


# ============================================================
# STEP 20: DETECT ROOM
# ============================================================

def detect_room(query):

    normalized = normalize_text(
        query
    )

    # Examples:
    # room 1
    # room1
    # lab 1
    # lab1
    # hall
    # library

    match = re.search(
        r"\b(room|lab)\s*-?\s*(\d+)\b",
        normalized
    )

    if match:

        return (
            match.group(1)
            + match.group(2)
        )

    if "library" in normalized:
        return "library"

    if "hall" in normalized:
        return "hall"

    return None


# ============================================================
# STEP 21: DETECT INSTRUCTOR
# ============================================================

def detect_instructor(query):

    normalized_query = normalize_text(
        query
    )

    instructors = get_all_instructors()

    # First try exact/partial instructor matching
    for instructor in instructors:

        normalized_instructor = normalize_text(
            instructor
        )

        if normalized_instructor in normalized_query:

            return instructor

    # Try last name
    query_words = set(
        normalized_query.split()
    )

    for instructor in instructors:

        instructor_words = normalize_text(
            instructor
        ).split()

        if len(instructor_words) >= 2:

            last_name = instructor_words[-1]

            if last_name in query_words:

                return instructor

    return None


# ============================================================
# STEP 22: DETECT SUBJECT
# ============================================================

def detect_subject(query):

    normalized_query = normalize_text(
        query
    )

    subjects = get_all_subjects()

    # Longest subjects first
    subjects = sorted(
        subjects,
        key=len,
        reverse=True
    )

    for subject in subjects:

        normalized_subject = normalize_text(
            subject
        )

        if normalized_subject in normalized_query:

            return subject

    # Common aliases
    aliases = {
        "ai": "Artificial Intelligence",
        "nlp": "Natural Language Processing",
        "oop": "Object Oriented Programming",
        "dld": "Digital Logic Design",
        "database": "Database Management Systems"
    }

    for alias, subject in aliases.items():

        if re.search(
            rf"\b{re.escape(alias)}\b",
            normalized_query
        ):

            return subject

    return None


# ============================================================
# STEP 23: FACULTY SUPPORT AGENT
# ============================================================

def faculty_support_agent(user_input):

    query = user_input.strip()

    if not query:

        return (
            "Please enter a question."
        )

    query_lower = normalize_text(
        query
    )


    # ========================================================
    # 1. TODAY
    # ========================================================

    if "today" in query_lower:

        # If user also mentions section
        section = detect_section(query)

        if section:

            results = find_by_section(
                section
            )

            # Filter today's result
            today = datetime.now().strftime(
                "%A"
            )

            today_results = [
                day
                for day in results
                if normalize_text(
                    day["day"]
                ) == normalize_text(today)
            ]

            if today_results:

                return (
                    f"📋 {section} schedule for today:\n"
                    + format_schedule(
                        today_results
                    )
                )

            return (
                f"No classes found for "
                f"{section} today."
            )

        # Otherwise show today's timetable
        result = find_today_schedule()

        if result:

            return (
                "📋 Today's Schedule:\n"
                + format_schedule(result)
            )

        return "No classes found today."


    # ========================================================
    # 2. SECTION QUERY
    # ========================================================

    section = detect_section(query)

    if section:

        results = find_by_section(
            section
        )

        if results:

            # If a day is also mentioned,
            # filter by that day.

            day = detect_day(query)

            if day and day != "today":

                results = [
                    item
                    for item in results
                    if normalize_text(
                        item["day"]
                    ) == normalize_text(day)
                ]

            if results:

                return (
                    f"📋 Schedule for {section}:\n"
                    + format_schedule(results)
                )

            return (
                f"No classes found for "
                f"{section} on {day}."
            )

        return (
            f"No schedule found for "
            f"{section}."
        )


    # ========================================================
    # 3. INSTRUCTOR QUERY
    # ========================================================

    instructor = detect_instructor(
        query
    )

    instructor_words = [
        "instructor",
        "teacher",
        "faculty",
        "professor",
        "prof",
        "dr",
        "doctor",
        "mr",
        "mrs",
        "miss",
        "classes taught",
        "teach"
    ]

    instructor_question = any(
        word in query_lower
        for word in instructor_words
    )

    if instructor and instructor_question:

        results = find_by_instructor(
            instructor
        )

        if results:

            return (
                f"📋 Classes taught by "
                f"{instructor}:\n"
                + format_schedule(results)
            )

        return (
            f"No classes found for "
            f"{instructor}."
        )


    # ========================================================
    # 4. ROOM QUERY
    # ========================================================

    room = detect_room(query)

    room_question = any(
        word in query_lower
        for word in [
            "room",
            "lab",
            "hall",
            "library"
        ]
    )

    if room and room_question:

        results = find_by_room(
            room
        )

        if results:

            return (
                f"📋 Classes in {room}:\n"
                + format_schedule(results)
            )

        return (
            f"No classes found in {room}."
        )


    # ========================================================
    # 5. SUBJECT QUERY
    # ========================================================

    subject = detect_subject(
        query
    )

    subject_question = any(
        word in query_lower
        for word in [
            "subject",
            "course",
            "class",
            "classes",
            "teach",
            "teacher",
            "instructor",
            "when",
            "where"
        ]
    )

    if subject and subject_question:

        results = find_by_subject(
            subject
        )

        if results:

            return (
                f"📋 Schedule for "
                f"{subject}:\n"
                + format_schedule(results)
            )

        return (
            f"No classes found for "
            f"{subject}."
        )


    # ========================================================
    # 6. DIRECT DAY QUERY
    # ========================================================

    day = detect_day(query)

    if day:

        if day == "today":

            result = find_today_schedule()

        else:

            result = find_by_day(
                day
            )

        if result:

            return (
                f"📋 {day.title()} Schedule:\n"
                + format_schedule(result)
            )

        return (
            f"No classes found for "
            f"{day.title()}."
        )


    # ========================================================
    # 7. GEMINI FALLBACK
    # ========================================================

    if client:

        # Build useful timetable context
        #
        # We do NOT need to send the entire JSON
        # for every simple query. Gemini is mainly
        # used for general faculty questions here.

        sections = get_all_sections()
        subjects = get_all_subjects()

        prompt = f"""
You are a Faculty Support Assistant for
Federal Urdu University of Arts, Science and Technology,
Karachi.

You are helping faculty members and students.

Available sections:
{", ".join(sections)}

Available subjects:
{", ".join(subjects)}

The timetable contains Monday through Friday.

User question:
{user_input}

Instructions:

Be helpful, concise, and accurate.

Answer questions about the timetable, teachers/instructors, subjects/courses, sections, rooms, schedules, faculty workload, and course allocation using only the information available in the system data.

If the user asks about timetable information but has not provided enough details to identify the requested information, ask them for the required details, such as:

day
section
instructor
room
subject/course.

Do not invent, assume, or fabricate any timetable, teacher, course, section, room, workload, or allocation information.

If the user asks for information that should be available from the system data, but the requested information is missing, unavailable, or not found in the system, respond exactly with:

"Sorry for inconvenience it seems like the data you are looking is not inserted on the system now"

When information is missing from the system, DO NOT tell the user to contact:

a faculty member
instructor
teacher
department
department office
administrator
staff member
faculty office
or any other person or office.

Do not provide alternative sources, contact information, or referrals when requested system data is unavailable.

For general questions that are NOT asking for specific system data (for example, greetings, general questions, explanations, how the timetable assistant works, or general educational questions), answer the user normally and helpfully. Do NOT use the missing-data response simply because the question is outside the timetable system.

If the user asks how the timetable assistant works, explain its functionality clearly.

If the user asks about a topic unrelated to the available system data, answer normally when you can. Do not automatically respond with the missing-data message.

Never claim that information exists in the system unless it is actually present in the provided data.

Keep responses clear, professional, and concise.

Answer the user:
"""

        try:

            response = client.models.generate_content(
                model=MODEL,
                contents=prompt
            )

            return response.text

        except Exception as e:

            print(
                f"\n⚠️ Gemini error: {e}"
            )


    # ========================================================
    # 8. FALLBACK RESPONSE
    # ========================================================

    return """
I can help you with the university timetable.

Try questions such as:

📅 "Show me Monday schedule"

📅 "What classes are on Tuesday?"

📚 "Show me BS2A schedule"

📚 "What does BS6A have on Wednesday?"

👨‍🏫 "What classes does Dr Khalid Sheikh teach?"

👨‍🏫 "Where does Dr Uzma Afzal teach?"

🏫 "What classes are in Lab1?"

🏫 "What is happening in Room12?"

📖 "Who teaches Artificial Intelligence?"

📖 "When is Database Management Systems?"

📖 "Show me NLP classes"

📅 "What does BS4B have today?"
"""


# ============================================================
# STEP 24: TEST FUNCTIONS
# ============================================================

print()
print("=" * 70)
print("🧪 TESTING TIMETABLE FUNCTIONS")
print("=" * 70)


# Test section
test_section = "BS2A"

section_result = find_by_section(
    test_section
)

print(
    f"\n📚 Testing section: {test_section}"
)

print(
    f"Days found: "
    f"{len(section_result)}"
)


# Test instructor
test_instructor = "Khalid Sheikh"

instructor_result = find_by_instructor(
    test_instructor
)

print(
    f"\n👨‍🏫 Testing instructor: "
    f"{test_instructor}"
)

print(
    f"Days found: "
    f"{len(instructor_result)}"
)


# Test room
test_room = "Lab1"

room_result = find_by_room(
    test_room
)

print(
    f"\n🏫 Testing room: {test_room}"
)

print(
    f"Days found: "
    f"{len(room_result)}"
)


# Test subject
test_subject = "Artificial Intelligence"

subject_result = find_by_subject(
    test_subject
)

print(
    f"\n📖 Testing subject: "
    f"{test_subject}"
)

print(
    f"Days found: "
    f"{len(subject_result)}"
)


print()
print("=" * 70)
print("✅ TESTING COMPLETE")
print("=" * 70)


# ============================================================
# STEP 25: DISPLAY AVAILABLE DATA
# ============================================================

print()
print("📚 Available Sections:")
print(", ".join(get_all_sections()))

print()
print(
    f"👨‍🏫 Total instructors: "
    f"{len(get_all_instructors())}"
)

print(
    f"📖 Total subjects: "
    f"{len(get_all_subjects())}"
)


# ============================================================
# STEP 26: MAIN CHAT LOOP
# ============================================================

print()
print("=" * 70)
print("🎓 FEDERAL URDU UNIVERSITY")
print("FACULTY SUPPORT AGENT")
print("=" * 70)

print()
print("I can help with:")

print(
    "  📅 Schedule by day"
)

print(
    "  📚 Schedule by section"
)

print(
    "  👨‍🏫 Classes by instructor"
)

print(
    "  🏫 Classes by room"
)

print(
    "  📖 Classes by subject"
)

print(
    "  🤖 General faculty questions"
)

print()
print("Examples:")
print(
    '  "Show me Monday schedule"'
)

print(
    '  "What does BS2A have on Tuesday?"'
)

print(
    '  "What classes does Dr Khalid Sheikh teach?"'
)

print(
    '  "Who teaches Artificial Intelligence?"'
)

print(
    '  "What classes are in Lab1?"'
)

print(
    '  "Show me BS4 schedule"'
)

print(
    '  "What does BS6A have today?"'
)

print()
print("Type 'exit' to stop.")
print("=" * 70)


while True:

    try:

        user = input(
            "\nYou: "
        ).strip()

        # Exit
        if user.lower() in [
            "exit",
            "quit",
            "bye"
        ]:

            print()
            print(
                "👋 Goodbye! Have a great day!"
            )

            break


        # Process query
        reply = faculty_support_agent(
            user
        )


        print()
        print("-" * 70)
        print("🤖 Agent:")
        print(reply)
        print("-" * 70)


    except KeyboardInterrupt:

        print()
        print(
            "\n👋 Agent stopped."
        )

        break


    except Exception as e:

        print()
        print(
            "❌ An error occurred:"
        )

        print(
            f"{type(e).__name__}: {e}"
        )

        print()
        print(
            "Please try rephrasing your question."
        )

🎓 FEDERAL URDU UNIVERSITY
Faculty Support Agent
✅ Timetable loaded successfully!
📅 Number of days: 5

📊 TIMETABLE STRUCTURE
First day: Monday
Classes type: <class 'list'>
Number of sections: 18
First section: BS2A

✅ Gemini AI initialized successfully!

🧪 TESTING TIMETABLE FUNCTIONS

📚 Testing section: BS2A
Days found: 5

👨‍🏫 Testing instructor: Khalid Sheikh
Days found: 4

🏫 Testing room: Lab1
Days found: 5

📖 Testing subject: Artificial Intelligence
Days found: 5

✅ TESTING COMPLETE

📚 Available Sections:
BS2A, BS2B, BS2C, BS2D, BS2E, BS2F, BS4A, BS4B, BS4C, BS4D, BS6A, BS6B, BS6C, BS6D, BS7, BS8A, BS8B, BS8C

👨‍🏫 Total instructors: 34
📖 Total subjects: 23

🎓 FEDERAL URDU UNIVERSITY
FACULTY SUPPORT AGENT

I can help with:
  📅 Schedule by day
  📚 Schedule by section
  👨‍🏫 Classes by instructor
  🏫 Classes by room
  📖 Classes by subject
  🤖 General faculty questions

Examples:
  "Show me Monday schedule"
  "What does BS2A have on Tuesday?"
  "What classes does Dr Khalid Sheikh teach?"
